# 03 · Record read-only telemetry to a new CSV

The default is ten seconds at a requested one-second interval. Use 600 seconds for the commissioning read-only record. A unique backend/UTC-named CSV goes to `OUTPUT_DIR`; existing files are refused. Each row is flushed, and the first read failure adds blank measurements with an error and ends the run.

Read [the tutorial guide](README.md) and [physical commissioning](../docs/commissioning.md) first. Use one serial owner. These notebook outputs are intentionally empty; no bench result is implied.


## Environment

From the repository root, run:

```powershell
uv sync --locked
uv run --locked --with jupyterlab --with ipykernel python -m jupyterlab
```

Choose that environment's Python kernel and run top to bottom. Restart the kernel after driver changes. The next cell opens no serial port.


In [ ]:
import sys
from pathlib import Path

# Prefer this checkout when launched from the repository root or examples/.
REPO_ROOT = Path.cwd().resolve()
for candidate in (REPO_ROOT, REPO_ROOT.parent):
    if (candidate / "src/tark_nrc/controller.py").is_file():
        REPO_ROOT = candidate
        sys.path.insert(0, str(candidate / "src"))
        break
print(f"Python: {sys.executable}")
print(f"Repository/config base: {REPO_ROOT}")

## Imports


In [ ]:
from datetime import UTC, datetime
from math import isfinite
from pathlib import Path
from time import monotonic, sleep

from tark_nrc import CsvRecorder, create_chiller

## Choose connection and recording settings

Set `CONFIG_PATH` to your verified NRC400 profile. This keeps USB adapter/unit identity bindings and verified serial framing/timeouts together. The current `[control]` section accepts only `expected_serial_number`; remove old `allow_writes`, `min_c` and `max_c` entries. The API has no write-enable switch, target range or state-based control lock. Choose targets for the actual coolant and experiment. There is no guessed COM port. `SIMULATE = True` ignores the hardware profile and opens no physical port; each run creates a fresh simulated device with no shared state. Review all operation settings before continuing.


In [ ]:
CONFIG_PATH = REPO_ROOT / "config.toml"
SIMULATE = False
DURATION_S = 10.0  # Use 600.0 for the commissioning ten-minute read-only record.
INTERVAL_S = 1.0  # Minimum start-to-start interval; serial transactions may take longer.
OUTPUT_DIR = REPO_ROOT / "outputs"

## Connect, record sequential samples and release

Run this entire operation cell together: connection and cleanup share one context/finally block. Ctrl+C/Interrupt attempts cleanup while allowing Python to finish. Do not split the cell, kill the kernel during I/O, or rerun setup over an unfinished connection. Errors remain visible. Closing the port retains the equipment's target, loop and pump state.


In [ ]:
for name, value in (("DURATION_S", DURATION_S), ("INTERVAL_S", INTERVAL_S)):
    if isinstance(value, bool) or not isfinite(value) or value <= 0:
        raise ValueError(f"{name} must be a positive finite number of seconds")

chiller = create_chiller(None if SIMULATE else CONFIG_PATH, simulate=SIMULATE)
stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S_%fZ")
csv_path = OUTPUT_DIR / f"{chiller.transport.backend}_{stamp}.csv"
with chiller:
    print(f"Backend: {chiller.transport.backend}; identity: {chiller.identity}")
    print(f"Recording to: {csv_path.resolve()}")
    # CsvRecorder uses exclusive creation and flushes each row; no overwrites.
    with CsvRecorder(csv_path, chiller.identity, chiller.transport.backend) as recorder:
        deadline = monotonic() + DURATION_S
        while monotonic() < deadline:
            started = monotonic()
            try:
                sample = chiller.read_sample()
            except (OSError, RuntimeError) as exc:
                recorder.record(error=str(exc))  # Blank measurements for this failed read.
                raise  # Stop at the first failed read; no automatic reconnect/retry.
            recorder.record(sample)
            print(
                f"{sample.timestamp_utc} | {sample.temperature_c:.2f} C | "
                f"target {sample.setpoint_c:.2f} C | {sample.control_loop}/{sample.pump}"
            )
            remaining = min(INTERVAL_S - (monotonic() - started), deadline - monotonic())
            sleep(max(0.0, remaining))
        print(f"Saved {recorder.rows} rows.")

print("CSV and connection closed; equipment state retained.")

## Check results and finish

Retain the terminal outcome and CSV. Check identity/backend, UTC timestamps, gaps and the error column. Slow transactions can overrun interval/duration; no exact row count is guaranteed. Disk errors may prevent an error row. Interruption leaves a partial record. Recording does not start or stop the chiller.

**Software connection closure does not stop the chiller or pump.** Follow the physical shutdown procedure separately and confirm panel state. A forced kernel termination cannot guarantee cleanup. See [the hardware test record checklist](README.md#6-record-a-hardware-test-result) for evidence to retain.
